In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


In [2]:
DATA_PATH = r"C:\Users\admin\Desktop\Forex_algo\code\Data\EUR_USD_H1.parquet"

df = pd.read_parquet(DATA_PATH)

print(df.head())
print(df.tail())
print("Time range:", df["time"].min(), "→", df["time"].max())
print("Shape:", df.shape)


                       time  volume    mid_o    mid_h    mid_l    mid_c  \
0 2016-01-07 00:00:00+00:00     542  1.07764  1.07832  1.07744  1.07778   
1 2016-01-07 01:00:00+00:00    3167  1.07776  1.08100  1.07748  1.08029   
2 2016-01-07 02:00:00+00:00    1567  1.08026  1.08176  1.07996  1.08152   
3 2016-01-07 03:00:00+00:00     914  1.08156  1.08257  1.08150  1.08187   
4 2016-01-07 04:00:00+00:00     649  1.08190  1.08256  1.08156  1.08236   

     bid_o    bid_h    bid_l    bid_c    ask_o    ask_h    ask_l    ask_c  
0  1.07757  1.07823  1.07735  1.07770  1.07772  1.07840  1.07752  1.07787  
1  1.07768  1.08092  1.07740  1.08020  1.07784  1.08109  1.07756  1.08038  
2  1.08018  1.08169  1.07987  1.08144  1.08035  1.08184  1.08005  1.08159  
3  1.08147  1.08249  1.08142  1.08178  1.08164  1.08265  1.08157  1.08196  
4  1.08182  1.08247  1.08147  1.08228  1.08199  1.08264  1.08163  1.08245  
                           time  volume    mid_o    mid_h    mid_l    mid_c  \
61468 2025-11-

In [3]:
# Make sure sorted by time (oldest → newest)
df = df.sort_values("time").reset_index(drop=True)

# Ensure 'time' is datetime (usually already is, but just in case)
df["time"] = pd.to_datetime(df["time"])

# Drop any fully broken rows
df = df.dropna().reset_index(drop=True)

df.head()


,time,volume,mid_o,mid_h,mid_l,mid_c,bid_o,bid_h,bid_l,bid_c,ask_o,ask_h,ask_l,ask_c
0,2016-01-07 00:00:00+00:00,542,1.07764,1.07832,1.07744,1.07778,1.07757,1.07823,1.07735,1.07770,1.07772,1.07840,1.07752,1.07787
1,2016-01-07 01:00:00+00:00,3167,1.07776,1.08100,1.07748,1.08029,1.07768,1.08092,1.07740,1.08020,1.07784,1.08109,1.07756,1.08038
2,2016-01-07 02:00:00+00:00,1567,1.08026,1.08176,1.07996,1.08152,1.08018,1.08169,1.07987,1.08144,1.08035,1.08184,1.08005,1.08159
3,2016-01-07 03:00:00+00:00,914,1.08156,1.08257,1.08150,1.08187,1.08147,1.08249,1.08142,1.08178,1.08164,1.08265,1.08157,1.08196
4,2016-01-07 04:00:00+00:00,649,1.08190,1.08256,1.08156,1.08236,1.08182,1.08247,1.08147,1.08228,1.08199,1.08264,1.08163,1.08245


In [4]:
# Close price (mid close)
df["close"] = df["mid_c"]

# Returns
df["ret_1"]  = df["close"].pct_change(1)   # last 1 hour return
df["ret_3"]  = df["close"].pct_change(3)   # last 3 hours
df["ret_6"]  = df["close"].pct_change(6)   # last 6 hours
df["ret_12"] = df["close"].pct_change(12)  # last 12 hours

# Volatility of 1h returns
df["vol_10"] = df["ret_1"].rolling(10).std()
df["vol_20"] = df["ret_1"].rolling(20).std()

# Moving averages of price
df["ma_20"] = df["close"].rolling(20).mean()
df["ma_50"] = df["close"].rolling(50).mean()

# Distance from price to MAs (relative)
df["dist_ma20"] = df["close"] / df["ma_20"] - 1
df["dist_ma50"] = df["close"] / df["ma_50"] - 1

# Drop initial NaNs created by rolling windows
df = df.dropna().reset_index(drop=True)

print(df[["time", "close", "volume", "ret_1", "ret_3", "ret_6", "vol_10", "ma_20", "dist_ma20"]].head())


                       time    close  volume     ret_1     ret_3     ret_6  \
0 2016-01-11 01:00:00+00:00  1.09378    2011 -0.000091 -0.000585  0.003238   
1 2016-01-11 02:00:00+00:00  1.09209     809 -0.001545 -0.001527  0.000137   
2 2016-01-11 03:00:00+00:00  1.09142     381 -0.000614 -0.002249 -0.001107   
3 2016-01-11 04:00:00+00:00  1.09142     251  0.000000 -0.002158 -0.002741   
4 2016-01-11 05:00:00+00:00  1.09180     343  0.000348 -0.000266 -0.001792   

     vol_10     ma_20  dist_ma20  
0  0.001103  1.089457   0.003968  
1  0.000951  1.089754   0.002144  
2  0.000997  1.089891   0.001402  
3  0.000980  1.090102   0.001209  
4  0.000975  1.090327   0.001351  


In [5]:
HORIZON = 3   # predict movement over next 3 candles (3 hours)
TH = 0.0005   # 0.05% move ~ about 5 pips on EURUSD

# Future close after HORIZON
df["future_close"] = df["close"].shift(-HORIZON)

# Future return over HORIZON
df["ret_future"] = (df["future_close"] - df["close"]) / df["close"]

# Initialize direction as NaN
df["direction"] = np.nan

# UP = 1, DOWN = 0
df.loc[df["ret_future"] >  TH, "direction"] = 1
df.loc[df["ret_future"] < -TH, "direction"] = 0

# Drop candles where move is too small (noise zone)
df = df.dropna(subset=["direction"]).reset_index(drop=True)
df["direction"] = df["direction"].astype(int)

print(df["direction"].value_counts(normalize=True))
print(df[["time", "close", "ret_future", "direction"]].head(10))


direction
1    0.503034
0    0.496966
Name: proportion, dtype: float64
                       time    close  ret_future  direction
0 2016-01-11 01:00:00+00:00  1.09378   -0.002158          0
1 2016-01-11 03:00:00+00:00  1.09142    0.001631          1
2 2016-01-11 04:00:00+00:00  1.09142   -0.001246          0
3 2016-01-11 05:00:00+00:00  1.09180   -0.003068          0
4 2016-01-11 06:00:00+00:00  1.09320   -0.004226          0
5 2016-01-11 08:00:00+00:00  1.08845    0.001314          1
6 2016-01-11 09:00:00+00:00  1.08858    0.001213          1
7 2016-01-11 10:00:00+00:00  1.09007   -0.001780          0
8 2016-01-11 12:00:00+00:00  1.08990   -0.004367          0
9 2016-01-11 14:00:00+00:00  1.08982   -0.001560          0


In [6]:
FEATURES = [
    # raw candle info
    "mid_o", "mid_h", "mid_l", "mid_c", "volume",
    # engineered features
    "ret_1", "ret_3", "ret_6", "ret_12",
    "vol_10", "vol_20",
    "dist_ma20", "dist_ma50",
]

len(FEATURES), FEATURES


(13,
 ['mid_o',
  'mid_h',
  'mid_l',
  'mid_c',
  'volume',
  'ret_1',
  'ret_3',
  'ret_6',
  'ret_12',
  'vol_10',
  'vol_20',
  'dist_ma20',
  'dist_ma50'])

In [7]:
SEQ_LEN = 50  # window size: 50 hours

def create_sequences(df, features, seq_len):
    X, y = [], []
    data = df[features].values
    target = df["direction"].values

    for i in range(len(df) - seq_len):
        # window [i, i+seq_len)
        X.append(data[i : i + seq_len])
        # label is at the end of the window
        y.append(target[i + seq_len])

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int64)

X, y = create_sequences(df, FEATURES, SEQ_LEN)

print("X shape:", X.shape)  # (samples, seq_len, n_features)
print("y shape:", y.shape)
print("Class balance:", np.bincount(y) / len(y))


X shape: (38188, 50, 13)
y shape: (38188,)
Class balance: [0.49685765 0.50314235]


In [8]:
N = len(X)
train_size = int(N * 0.7)
val_size   = int(N * 0.15)
test_size  = N - train_size - val_size

X_train = X[:train_size]
y_train = y[:train_size]

X_val   = X[train_size: train_size + val_size]
y_val   = y[train_size: train_size + val_size]

X_test  = X[train_size + val_size:]
y_test  = y[train_size + val_size:]

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)


Train: (26731, 50, 13) Val: (5728, 50, 13) Test: (5729, 50, 13)


In [9]:
class PriceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_ds = PriceDataset(X_train, y_train)
val_ds   = PriceDataset(X_val, y_val)
test_ds  = PriceDataset(X_test, y_test)

BATCH_SIZE = 128

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)


In [10]:
n_features = len(FEATURES)
seq_len = SEQ_LEN

class PriceCNN(nn.Module):
    def __init__(self, n_features, seq_len):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels=n_features, out_channels=32, kernel_size=3, padding=1)
        self.bn1   = nn.BatchNorm1d(32)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.bn2   = nn.BatchNorm1d(64)
        self.relu  = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        
        self.fc   = nn.Sequential(
            nn.Linear(64 * seq_len, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)  # output: logit for binary classification
        )
        
    def forward(self, x):
        # x: (batch, seq_len, features) -> (batch, features, seq_len)
        x = x.permute(0, 2, 1)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.dropout(x)
        x = x.reshape(x.size(0), -1)
        x = self.fc(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = PriceCNN(n_features=n_features, seq_len=seq_len).to(device)
model


Using device: cuda


PriceCNN(
  (conv1): Conv1d(13, 32, kernel_size=(3,), stride=(1,), padding=(1,))
  (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv1d(32, 64, kernel_size=(3,), stride=(1,), padding=(1,))
  (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU()
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Sequential(
    (0): Linear(in_features=3200, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=1, bias=True)
  )
)

In [11]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def train_one_epoch(model, loader):
    model.train()
    total_loss = 0.0
    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.float().to(device)  # BCE expects float (0.0 / 1.0)

        optimizer.zero_grad()
        logits = model(X_batch).squeeze(1)         # (batch,)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X_batch.size(0)

    return total_loss / len(loader.dataset)

def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.float().to(device)

            logits = model(X_batch).squeeze(1)
            loss = criterion(logits, y_batch)

            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).long().cpu().numpy()

            total_loss += loss.item() * X_batch.size(0)
            all_preds.append(preds)
            all_labels.append(y_batch.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels).astype(int)
    acc = (all_preds == all_labels).mean()
    return avg_loss, acc, all_labels, all_preds


In [12]:
EPOCHS = 20

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader)
    val_loss, val_acc, _, _ = evaluate(model, val_loader)
    print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")


Epoch 01 | Train Loss: 0.6959 | Val Loss: 0.6935 | Val Acc: 0.4953
Epoch 02 | Train Loss: 0.6926 | Val Loss: 0.6949 | Val Acc: 0.4916
Epoch 03 | Train Loss: 0.6921 | Val Loss: 0.6941 | Val Acc: 0.5045
Epoch 04 | Train Loss: 0.6918 | Val Loss: 0.6938 | Val Acc: 0.4984
Epoch 05 | Train Loss: 0.6909 | Val Loss: 0.6948 | Val Acc: 0.4965
Epoch 06 | Train Loss: 0.6902 | Val Loss: 0.6944 | Val Acc: 0.5052
Epoch 07 | Train Loss: 0.6895 | Val Loss: 0.6971 | Val Acc: 0.5080
Epoch 08 | Train Loss: 0.6881 | Val Loss: 0.6959 | Val Acc: 0.5037
Epoch 09 | Train Loss: 0.6876 | Val Loss: 0.6966 | Val Acc: 0.5147
Epoch 10 | Train Loss: 0.6853 | Val Loss: 0.7014 | Val Acc: 0.5044
Epoch 11 | Train Loss: 0.6842 | Val Loss: 0.6994 | Val Acc: 0.5056
Epoch 12 | Train Loss: 0.6826 | Val Loss: 0.7015 | Val Acc: 0.5005
Epoch 13 | Train Loss: 0.6800 | Val Loss: 0.7073 | Val Acc: 0.4909
Epoch 14 | Train Loss: 0.6779 | Val Loss: 0.7089 | Val Acc: 0.4998
Epoch 15 | Train Loss: 0.6747 | Val Loss: 0.7094 | Val Acc: 0.

In [13]:
test_loss, test_acc, y_true, y_pred = evaluate(model, test_loader)
print(f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")

print("\nClassification report:")
print(classification_report(y_true, y_pred))

print("\nConfusion matrix:")
print(confusion_matrix(y_true, y_pred))


Test Loss: 0.7711, Test Acc: 0.5051

Classification report:
              precision    recall  f1-score   support

           0       0.50      0.49      0.49      2809
           1       0.51      0.52      0.52      2920

    accuracy                           0.51      5729
   macro avg       0.50      0.50      0.50      5729
weighted avg       0.51      0.51      0.51      5729


Confusion matrix:
[[1371 1438]
 [1397 1523]]
